# MiMo-Audio-7B-Instruct  -  Audio Event Evaluation

Evaluates XiaomiMiMo/MiMo-Audio-7B-Instruct on 9 surveillance audio clips.
Compares predictions against ground-truth labels and computes performance metrics.

**Model**: XiaomiMiMo/MiMo-Audio-7B-Instruct (4-bit quantized, 8-channels audio)
**Architecture**: Qwen2-7B + 16-layer local transformer + MiMo-Audio-Tokenizer (20-level RVQ, 24kHz)
**VRAM**: ~8.0 GB (4-bit) + audio tokenizer


In [ ]:
import json, os, re, sys, warnings
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings("ignore")
%matplotlib inline

# --- auto-detect project root ---
_root = Path.cwd()
for _p in [_root] + list(_root.parents):
    if (_p / ".gitignore").exists():
        PROJECT_ROOT = _p; break
else:
    PROJECT_ROOT = _root
del _root, _p
# --------------------------------

MIMO_DIR = PROJECT_ROOT / "experiments/MiMo-Audio"

import re

_RE_KEEP = re.compile(r"[^a-zA-Z0-9_ ]+")

def _stem(word: str) -> str:
    w = word.lower().strip("_.,;:!?")
    for suf in ["ing", "ed", "ly", "s", "es", "ies", "ness", "tion"]:
        if w.endswith(suf) and len(w) > len(suf) + 2:
            return w[:-len(suf)]
    return w

_GT_KEYWORDS = {
    "scream":          {"scream"},
    "shout":           {"shout", "yell"},
    "impact":          {"impact", "thump", "thud", "bang", "slam", "smash", "crash", "punch", "hit"},
    "gunshot_or_explosion": {"gunshot", "gunfire", "artillery_fire", "artillery", "explosion", "explosive", "firework", "boom"},
        "tire_squeal":     {"tire", "tyre", "tire_squeal", "screech", "squeal"},
    "skidding":        {"skidding", "skid"},
    "glass_breaking":  {"glass", "glass_breaking", "shatter", "shattering"},
    "horn":        {"horn", "honk", "honking"},
}

_SYNONYM_LOOKUP: dict[str, str] = {}
for gt, kws in _GT_KEYWORDS.items():
    for kw in kws:
        _SYNONYM_LOOKUP[kw] = gt
        _SYNONYM_LOOKUP[_stem(kw)] = gt

TIRE_SQUEAL_KWS = _GT_KEYWORDS["tire_squeal"]
HORN_KWS = _GT_KEYWORDS["horn"]

def _resolve_label(text: str) -> str | None:
    clean = _RE_KEEP.sub(" ", text or "").lower()
    clean = " ".join(clean.split())
    if not clean:
        return None
    tokens = [t.strip("_.,;:!?'\"") for t in re.split(r"[_ ,;:\-]+", clean)]
    tokens = [t for t in tokens if len(t) >= 2]
    if not tokens:
        return None
    # Priority: if any token is a tire_squeal keyword, resolve to tire_squeal
    for token in tokens:
        if token in TIRE_SQUEAL_KWS or _stem(token) in TIRE_SQUEAL_KWS:
            return "tire_squeal"
    # Priority: if any token is a horn keyword, resolve to horn
    for token in tokens:
        if token in HORN_KWS or _stem(token) in HORN_KWS:
            return "horn"
    fallback = tokens[0]
    for token in tokens:
        if token in _SYNONYM_LOOKUP:
            return _SYNONYM_LOOKUP[token]
        s = _stem(token)
        if s in _SYNONYM_LOOKUP:
            return _SYNONYM_LOOKUP[s]
        if token in _GT_LABELS:
            return token
    return fallback

AUDIO_DIR = PROJECT_ROOT / "data/audio/eval"

GROUND_TRUTH = {
    "scene1_scream_2.625-4.792.wav": "scream",
    "scene2_scream_2.667-5.083.wav": "scream",
    "scene3_scream_2.500-6.250.wav": "scream",
    "scene4_shout_0.000-1.500.wav": "shout",
    "scene4_impact_1.500-1.667.wav": "impact",
    "scene5_shout_0.000-1.833.wav": "shout",
    "scene5_impact_1.833-5.125.wav": "impact",
    "scene6_shout_0.000-2.167.wav": "shout",
    "scene6_impact_2.500-3.000.wav": "impact",
    "scene7_gunshot_or_explosion_4.000-4.375.wav": "gunshot_or_explosion",
    "scene7_gunshot_or_explosion_4.833-6.667.wav": "gunshot_or_explosion",
    "scene8_gunshot_or_explosion_2.500-3.208.wav": "gunshot_or_explosion",
    "scene9_gunshot_or_explosion_4.750-5.750.wav": "gunshot_or_explosion",
    "scene10_gunshot_or_explosion_0.583-4.167.wav": "gunshot_or_explosion",
    "scene11_gunshot_or_explosion_1.542-4.167.wav": "gunshot_or_explosion",
    "scene12_engine_3.875-5.500.wav": "engine",
    "scene12_tire_squeal_5.500-8.667.wav": "tire_squeal",
    "scene13_engine_3.958-7.875.wav": "engine",
    "scene13_tire_squeal_7.875-9.083.wav": "tire_squeal",
    "scene14_engine_3.667-7.042.wav": "engine",
    "scene14_tire_squeal_7.042-9.167.wav": "tire_squeal",
    "scene15_horn_0.792-3.042.wav": "horn",
    "scene15_impact_3.042-3.292.wav": "impact",
    "scene15_glass_breaking_3.292-4.542.wav": "glass_breaking",
    "scene16_skidding_1.292-2.208.wav": "skidding",
    "scene16_impact_2.208-2.458.wav": "impact",
    "scene16_glass_breaking_2.625-5.417.wav": "glass_breaking",
    "scene17_horn_1.250-3.083.wav": "horn",
    "scene17_skidding_2.208-3.083.wav": "skidding",
    "scene17_glass_breaking_3.083-4.583.wav": "glass_breaking"
}

CLASSES = sorted({v for v in GROUND_TRUTH.values()})
_GT_LABELS = sorted(set(GROUND_TRUTH.values()), key=len, reverse=True)


In [ ]:
print("Loading MiMo-Audio-7B-Instruct (4-bit)...", flush=True)

from mimo_audio.mimo_audio import MimoAudio
from transformers import BitsAndBytesConfig

model_path = str(MIMO_DIR / "models" / "MiMo-Audio-7B-Instruct")
tokenizer_path = str(MIMO_DIR / "models" / "MiMo-Audio-Tokenizer")

quant = None
if torch.cuda.is_available():
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

os.environ["TORCH_CUDAGRAPH_STOP_GROWTH"] = "1"
_model = MimoAudio(model_path, tokenizer_path, quantization_config=quant)
print(f"MiMo-Audio loaded on {_model.device}")

free, total = torch.cuda.mem_get_info()
print(f"VRAM: {(total-free)/1e9:.1f}/{total/1e9:.1f} GB")


In [ ]:
def _mimo_predict_one(wav_path: str) -> str:
    prompt = "What's that sound? Output one word or two words separated by underscore (e.g. first_second)."
    resp = _model.audio_understanding_sft(str(wav_path), prompt)
    return resp.strip().rstrip(".")

N_RUNS = 5

results = []
wav_files = sorted(AUDIO_DIR.glob("*.wav"))

import time
start_time = time.time()

for wf in wav_files:
    true_label = GROUND_TRUTH[wf.name]
    print(f"\n--- {wf.name} | True: {true_label} ---")

    votes = []
    raw_outputs = []
    for i in range(N_RUNS):
        resp = _mimo_predict_one(str(wf))
        raw_outputs.append(resp)
        gt = _resolve_label(resp)
        votes.append(gt or "none")
        print(f"  [{i+1}] {resp} -> {gt}")

    predicted_label = max(set(votes), key=votes.count)
    correct = predicted_label == true_label

    results.append({
        "file": wf.name,
        "true": true_label,
        "predicted": predicted_label,
        "votes": votes,
        "raw_outputs": raw_outputs,
        "correct": correct,
    })

    mark = "[OK]" if correct else "[NO]"
    print(f"  {mark} vote={predicted_label} (true={true_label})")

elapsed = time.time() - start_time
correct = sum(1 for r in results if r["correct"])
total = len(results)
print(f"\nAccuracy: {correct}/{total} ({100*correct//total}%)")
print(f"Total inference time: {elapsed:.1f}s ({elapsed/total:.2f}s per clip)")


In [ ]:
print("\n=== Classification Report ===\n")
y_true = [r["true"] for r in results]
y_pred = [r["predicted"] if r["predicted"] in CLASSES else "other" for r in results]

all_labels = sorted(set(y_true + [l for l in y_pred if l != "other"]))
if "other" in [l for l in y_pred]:
    all_labels.append("other")

print(classification_report(y_true, y_pred, labels=all_labels, zero_division=0))

print("\n--- Per-Class Metrics ---")
classes_in_data = sorted(set(y_true))
for cls in classes_in_data:
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p == cls)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t != cls and p == cls)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p != cls)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    support = tp + fn
    print(f"  {cls:12s}  prec={precision:.3f}  recall={recall:.3f}  f1={f1:.3f}  support={support}")


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=all_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=all_labels, yticklabels=all_labels)
plt.title("MiMo-Audio-7B  -  Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "experiments/notebooks/mimo_confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
# Accuracy per class
classes_in_data = sorted(set(y_true))
correct_by_class = {}
total_by_class = {}
for cls in classes_in_data:
    correct_by_class[cls] = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p == cls)
    total_by_class[cls] = sum(1 for t in y_true if t == cls)

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(classes_in_data))
accs = [correct_by_class[c] / total_by_class[c] * 100 for c in classes_in_data]
bars = ax.bar(x, accs, color=["#4CAF50" if a == 100 else "#FF9800" for a in accs])
ax.set_xticks(x)
ax.set_xticklabels(classes_in_data, rotation=45, ha="right")
ax.set_ylabel("Accuracy (%)")
ax.set_title("MiMo-Audio-7B  -  Per-Class Accuracy")
ax.set_ylim(0, 110)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{acc:.0f}%",
            ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "experiments/notebooks/mimo_per_class_accuracy.png", dpi=150)
plt.show()


In [ ]:
# Metrics Summary
from sklearn.metrics import precision_recall_fscore_support

mac_p, mac_r, mac_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
wgt_p, wgt_r, wgt_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
print("=== Metrics Summary ===")
print(f"  Macro     P={mac_p:.3f}  R={mac_r:.3f}  F1={mac_f1:.3f}")
print(f"  Weighted  P={wgt_p:.3f}  R={wgt_r:.3f}  F1={wgt_f1:.3f}")
print(f"  Accuracy: {correct}/{total} ({100*correct//total}%)")


In [ ]:
print(f"{'File':45s} {'True':12s} {'Predicted':12s} {'Votes':30s} {'Correct':8s}")
print("-" * 110)
for r in results:
    mark = "[OK]" if r["correct"] else "[NO]"
    votes_str = ", ".join(r["votes"])
    print(f"{r['file']:45s} {r['true']:12s} {r['predicted']:12s} {votes_str:30s} {mark:8s}")
print(f"\nOverall: {correct}/{total} ({100*correct//total}%)")
